In [0]:

CREATE DATABASE IF NOT EXISTS uber_db;
USE uber_db;

DROP TABLE IF EXISTS fact_trips;
DROP TABLE IF EXISTS dim_driver;
DROP TABLE IF EXISTS dim_passenger;
DROP TABLE IF EXISTS dim_location;
DROP TABLE IF EXISTS dim_date;


-- 1. Driver Dimension
CREATE TABLE dim_driver AS
SELECT DISTINCT 
    CAST(rand() * 1000 AS INT) AS driver_id,
    CASE WHEN rand() > 0.5 THEN 'Active' ELSE 'Inactive' END AS driver_status,
    current_date() AS registration_date
FROM samples.nyctaxi.trips LIMIT 100;


-- 2. Passenger Dimension
CREATE TABLE dim_passenger AS
SELECT DISTINCT 
    CAST(rand() * 5000 AS INT) AS passenger_id,
    CASE WHEN rand() > 0.3 THEN 'Gold' ELSE 'Standard' END AS membership_tier
FROM samples.nyctaxi.trips LIMIT 500;


-- 3. Location Dimension
CREATE TABLE dim_location AS
SELECT DISTINCT 
    CAST(pickup_zip AS STRING) AS zip_code,
    'New York' AS city,
    'NY' AS state
FROM samples.nyctaxi.trips 
WHERE pickup_zip IS NOT NULL 
LIMIT 50;


CREATE TABLE dim_date AS
SELECT DISTINCT 
    CAST(date_format(tpep_pickup_datetime, 'yyyyMMdd') AS INT) AS date_key,
    CAST(tpep_pickup_datetime AS DATE) AS full_date,
    EXTRACT(YEAR FROM tpep_pickup_datetime) AS year,
    EXTRACT(MONTH FROM tpep_pickup_datetime) AS month,
    EXTRACT(DAY FROM tpep_pickup_datetime) AS day,
    EXTRACT(DAYOFWEEK FROM tpep_pickup_datetime) AS day_of_week
FROM samples.nyctaxi.trips
LIMIT 365;

CREATE TABLE fact_trips AS
SELECT 
    monotonically_increasing_id() AS trip_id,
    CAST(date_format(tpep_pickup_datetime, 'yyyyMMdd') AS INT) AS date_key,
    CAST(rand() * 1000 AS INT) AS driver_id,
    CAST(rand() * 5000 AS INT) AS passenger_id,
    CAST(pickup_zip AS STRING) AS zip_code,
    trip_distance,
    fare_amount,
    unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime) AS trip_duration_seconds
FROM samples.nyctaxi.trips
LIMIT 10000;





WITH ranked_trips AS (
    SELECT 
        driver_id,
        trip_id,
        fare_amount,
        ROW_NUMBER() OVER (PARTITION BY driver_id ORDER BY fare_amount DESC) as fare_rank
    FROM fact_trips
)
SELECT * 
FROM ranked_trips 
WHERE fare_rank <= 3;


SELECT 
    d.full_date,
    SUM(f.fare_amount) AS daily_revenue,
    SUM(SUM(f.fare_amount)) OVER (ORDER BY d.full_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total_revenue
FROM fact_trips f
JOIN dim_date d ON f.date_key = d.date_key
GROUP BY d.full_date
ORDER BY d.full_date;

SELECT 
    CASE 
        WHEN trip_distance <= 2 THEN 'Short Trip (<= 2mi)'
        WHEN trip_distance > 2 AND trip_distance <= 10 THEN 'Medium Trip (2-10mi)'
        ELSE 'Long Trip (> 10mi)'
    END AS trip_category,
    COUNT(trip_id) AS total_trips,
    ROUND(AVG(fare_amount), 2) AS avg_fare
FROM fact_trips
GROUP BY 1;


SELECT 
    d.day_of_week,
    CASE d.day_of_week
        WHEN 1 THEN 'Sunday'
        WHEN 2 THEN 'Monday'
        WHEN 3 THEN 'Tuesday'
        WHEN 4 THEN 'Wednesday'
        WHEN 5 THEN 'Thursday'
        WHEN 6 THEN 'Friday'
        WHEN 7 THEN 'Saturday'
    END AS day_name,
    COUNT(f.trip_id) AS total_rides,
    ROUND(AVG(f.trip_duration_seconds / 60), 2) AS avg_duration_mins
FROM fact_trips f
JOIN dim_date d ON f.date_key = d.date_key
GROUP BY d.day_of_week
ORDER BY d.day_of_week;



SELECT 
    COALESCE(l.city, 'Unknown City') AS city,
    COALESCE(l.state, 'XX') AS state,
    COUNT(f.trip_id) AS total_trips
FROM fact_trips f
LEFT JOIN dim_location l ON f.zip_code = l.zip_code
GROUP BY l.city, l.state;

















